# 16.4 Protocols, Structural Typing, `Self` and `@overload`

**Prerequisites:** 16.3 Generics, 5.4 Duck Typing and Protocols, 5.1 OOPs  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- **Nominal vs structural** typing — inheritance versus shape
- `Protocol` as the type system's answer to duck typing (**5.4**)
- Choosing between `Protocol` and `ABC`
- How a checker reports a **signature mismatch**, member by member
- Generic protocols, and **callback protocols** for functions with named arguments
- 🔴 `runtime_checkable` checks **presence only** — not signatures, not types
- `Self` — the return type that follows subclasses
- `@overload` — one function, several precise signatures

---

## Two ways to say "this will do"

**5.4** made the case for duck typing: *if it has `.close()`, it is close-able*. The problem was
that this excellent idea had no way to be **checked** — you found out at runtime.

A type system can work either way:

| | **Nominal** typing | **Structural** typing |
|---|---|---|
| "Is X acceptable?" | does X *inherit* from it? | does X *have the right shape*? |
| Declared by | `class X(Base)` | nothing — the shape is enough |
| Python tool | `ABC` (**5.4**) | 🔴 **`Protocol`** |
| Works on code you cannot edit | ❌ | ✅ |
| Most languages | Java, C# | Go interfaces, TypeScript |

`Protocol` is duck typing that the checker can verify **before** you run anything. That is the
whole idea, and the rest of this notebook is what follows from it.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py164_"))


def write(name, source):
    """Write a script into the scratch directory and return its name."""
    (WORK / name).write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return name


def mypy(name, source=None, *flags):
    """Type-check a file with mypy and return its report."""
    if source is not None:
        write(name, source)
    done = subprocess.run(
        [sys.executable, "-m", "mypy", name,
         "--cache-dir", str(WORK / ".mypy_cache"),
         "--no-color-output", "--no-error-summary", *flags],
        cwd=WORK, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    report = (done.stdout + done.stderr).strip() or "(mypy found nothing to report)"
    return (f"$ mypy {name} {' '.join(flags)}".rstrip() + "\n" + "-" * 68 + "\n"
            + report + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


def python(name):
    done = subprocess.run([sys.executable, name], cwd=WORK, capture_output=True,
                          text=True, encoding="utf-8", errors="replace", timeout=60)
    return (f"$ python {name}\n" + "-" * 68 + "\n"
            + (done.stdout + done.stderr).strip() + "\n" + "-" * 68
            + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

## The case `Protocol` exists for

You depend on a class from a library. It has exactly the method you need. It does **not**
inherit from your base class, and you cannot make it — you do not own the code.

In [ ]:
print(mypy("nominal_structural.py", r"""
    from abc import ABC, abstractmethod
    from typing import Protocol


    class StoreABC(ABC):                       # NOMINAL: requires inheritance
        @abstractmethod
        def save(self, key: str, value: str) -> None: ...


    class StoreProto(Protocol):                # STRUCTURAL: requires the shape
        def save(self, key: str, value: str) -> None: ...


    class ThirdPartyStore:                     # a library class you cannot edit
        def save(self, key: str, value: str) -> None: ...


    def persist_nominal(store: StoreABC) -> None: ...
    def persist_structural(store: StoreProto) -> None: ...


    persist_structural(ThirdPartyStore())      # fine - the shape matches
    persist_nominal(ThirdPartyStore())         # 🔴 not a subclass of StoreABC
"""))

One error, and it is the `ABC`. The `Protocol` accepted a class that had never
heard of it.

### Which to reach for

| Use a **`Protocol`** when | Use an **`ABC`** when |
|---|---|
| implementers are outside your control | you own every implementer |
| you want to type an existing shape (`.read()`, `.close()`) | you want to **share code** via inheritance |
| the interface is small — one or two methods | you need `__init_subclass__`, registration, or template methods |
| you are typing a *callback* or a *parameter* | you want `isinstance` to be reliable |

🔴 The deciding question is usually: **do you need to give implementers behaviour, or just
describe them?** `ABC` can carry concrete methods; a `Protocol` only describes.

## When the shape does not quite match

A near-miss is more common than a total mismatch — the right method name with the wrong
signature. The checker reports it member by member.

In [ ]:
print(mypy("conflict.py", r"""
    from typing import Protocol


    class Closeable(Protocol):
        def close(self) -> None: ...


    class Connection:
        def close(self) -> None: ...


    class Report:
        def close(self, force: bool) -> None: ...      # extra required argument


    class Handle:
        def shutdown(self) -> None: ...                # wrong name entirely


    def release(resource: Closeable) -> None:
        resource.close()


    release(Connection())
    release(Report())
    release(Handle())
"""))

Read the `Report` error: mypy prints **Expected** and **Got** signatures side
by side. That is far more useful than "incompatible type", and it is why a small, precise
protocol beats a vague one.

`Handle` failed differently — the member is simply absent.

> 🔴 A protocol method with an extra **required** parameter cannot match, but one with an extra
> parameter that has a **default** can, since it is still callable with no arguments. That is
> the same substitutability rule as normal subclassing (**5.1**).

## Generic protocols

A protocol can take type parameters, using the PEP 695 syntax from **16.3**. This is how you
express "anything I can compare" or "anything I can iterate that yields `T`".

In [ ]:
print(mypy("generic_proto.py", r"""
    from typing import Protocol


    class Comparable[T](Protocol):
        def __lt__(self, other: T) -> bool: ...


    def smallest[T: Comparable](items: list[T]) -> T:
        return min(items)


    class Version:
        def __init__(self, major: int) -> None:
            self.major = major

        def __lt__(self, other: "Version") -> bool:
            return self.major < other.major


    class Opaque:
        pass


    reveal_type(smallest([3, 1, 2]))
    reveal_type(smallest(["b", "a"]))
    reveal_type(smallest([Version(2), Version(1)]))

    smallest([Opaque(), Opaque()])          # 🔴 no __lt__
"""))

`Version` satisfies `Comparable` because it defines `__lt__` — no inheritance,
no registration. `Opaque` does not, and the checker says so **before** `min()` would have
raised `TypeError` at runtime.

## Callback protocols

`Callable[[int, float], float]` can only describe **positional** parameters. The moment your
callback has a keyword argument, a default, or a meaningful parameter name, `Callable` cannot
express it.

A protocol with `__call__` can.

In [ ]:
print(mypy("callback.py", r"""
    from collections.abc import Callable
    from typing import Protocol


    class RetryPolicy(Protocol):
        def __call__(self, attempt: int, *, ceiling: float = ...) -> float: ...


    def run(policy: RetryPolicy) -> float:
        return policy(1, ceiling=30.0)


    def exponential(attempt: int, *, ceiling: float = 30.0) -> float:
        return min(2.0 ** attempt, ceiling)


    def fixed(attempt: int, *, ceiling: float = 30.0) -> float:
        return 1.0


    def wrong_shape(attempt: str) -> float:            # wrong type, no keyword
        return 1.0


    run(exponential)
    run(fixed)
    run(wrong_shape)

    # Callable accepts the assignment - but SILENTLY DISCARDS the keyword parameter:
    plain: Callable[[int], float] = exponential
    plain(1)
    plain(1, ceiling=30.0)          # 🔴 the information is gone
"""))

The protocol accepted both correctly-shaped functions and rejected the third,
naming the expected `__call__` signature.

🔴 Look at the last two lines. Assigning `exponential` to `Callable[[int], float]` **succeeded**
— it *is* callable with one positional `int` — but the keyword parameter was **silently
discarded**, so `plain(1, ceiling=30.0)` is now an error. `Callable` did not reject the
function; it forgot half of it.

Use a callback protocol whenever the callback has anything more interesting than positional
arguments.

## 🔴 `runtime_checkable` — and what it does **not** check

Adding `@runtime_checkable` lets a protocol be used with `isinstance()`. It is genuinely useful,
and it is much weaker than people assume.

> **`isinstance()` against a protocol checks that the members exist. Nothing else.**
> Not the signature. Not the types. Not the return value.

In [ ]:
print(python(write("runtime_checkable.py", r"""
    from typing import Protocol, runtime_checkable


    @runtime_checkable
    class Closeable(Protocol):
        def close(self) -> None: ...


    @runtime_checkable
    class HasName(Protocol):
        name: str


    class Report:
        def close(self, force: bool) -> None:      # WRONG signature
            pass


    class Job:
        def __init__(self) -> None:
            self.name = 12345                      # WRONG type


    print("Report has a close() with the wrong signature")
    print("   isinstance(Report(), Closeable) :", isinstance(Report(), Closeable))
    print()
    print("Job has a `name` attribute of the wrong type")
    print("   isinstance(Job(), HasName)      :", isinstance(Job(), HasName))
    print()
    print("Both are True. The checker would reject both.")
""")))

Both `True`. A class whose `close()` takes a required argument, and an object
whose `name` is an `int`, both pass `isinstance`.

| | Static check (`mypy`) | `isinstance` with `runtime_checkable` |
|---|---|---|
| Member exists | ✅ | ✅ |
| Signature matches | ✅ | ❌ |
| Attribute **type** matches | ✅ | ❌ |
| Cost | free | 🔴 slow — it inspects every member |

🔴 **Use `runtime_checkable` for a coarse "does this look roughly right" gate, never as
validation.** If the distinction matters at runtime, check the thing you actually care about,
or use the static check and trust it.

## `Self` — the return type that follows subclasses

A method returning "the same class as the receiver" — a fluent builder, `copy()`, a
`from_dict` classmethod — cannot be typed by naming the class, because subclasses would inherit
the wrong return type.

In [ ]:
print(mypy("selftype.py", r"""
    from typing import Self


    class QueryBuilder:
        def __init__(self) -> None:
            self.parts: list[str] = []

        def where(self, clause: str) -> Self:            # 🔴 Self, not "QueryBuilder"
            self.parts.append(clause)
            return self

        def clone(self) -> Self:
            copy = type(self)()
            copy.parts = list(self.parts)
            return copy


    class HardcodedBuilder:
        def where(self, clause: str) -> "HardcodedBuilder":     # the wrong way
            return self


    class TimedQueryBuilder(QueryBuilder):
        def timeout(self, seconds: float) -> Self:
            self.parts.append(f"TIMEOUT {seconds}")
            return self


    class TimedHardcoded(HardcodedBuilder):
        def timeout(self, seconds: float) -> "TimedHardcoded":
            return self


    reveal_type(TimedQueryBuilder().where("id = 1"))
    reveal_type(TimedHardcoded().where("id = 1"))

    # Chaining works only when the subclass type survives:
    TimedQueryBuilder().where("id = 1").timeout(5.0)
    TimedHardcoded().where("id = 1").timeout(5.0)
"""))

`Self` kept the subclass — revealed as `TimedQueryBuilder`, so `.timeout()`
chained fine. The hardcoded version collapsed to `HardcodedBuilder` and the chain broke.

This is the single most common reason a fluent API stops type-checking in a subclass, and the
fix is one word.

## `@overload` — several precise signatures for one function

Some functions genuinely change their return type based on their arguments. A single signature
can only describe the union, which pushes narrowing (**16.2**) onto every caller.

`@overload` declares the specific cases. The rules:

1. Write two or more `@overload` stubs with `...` as the body.
2. Follow them with **one** real implementation, which is **not** decorated.
3. The implementation's signature must be compatible with all of them, and is never seen by
   callers.

In [ ]:
print(mypy("overload.py", r"""
    from typing import overload


    @overload
    def get_setting(key: str) -> str | None: ...
    @overload
    def get_setting(key: str, default: str) -> str: ...

    def get_setting(key: str, default: str | None = None) -> str | None:
        return {"region": "eu"}.get(key, default)


    with_default = get_setting("region", "us")
    without = get_setting("missing")

    reveal_type(with_default)          # str - no None, because a default was given
    reveal_type(without)               # str | None

    print(with_default.upper())        # safe, no narrowing needed
    print(without.upper())             # 🔴 caught: may be None

    get_setting(123)                   # 🔴 no matching overload
"""))

That is the payoff: `get_setting("region", "us")` is **`str`**, so `.upper()`
needs no narrowing, while `get_setting("missing")` is `str | None` and the checker insists.
One function, two honest signatures.

### 🔴 The implementation must actually satisfy the stubs

The most common `@overload` mistake is an implementation that cannot return what a stub
promises. The checker catches it.

In [ ]:
print(mypy("bad_overload.py", r"""
    from typing import overload


    @overload
    def parse(raw: str) -> str: ...
    @overload
    def parse(raw: int) -> int: ...

    def parse(raw: str | int) -> str:          # 🔴 cannot return int
        return str(raw)


    @overload
    def coerce(raw: str) -> str: ...
    @overload
    def coerce(raw: int) -> int: ...

    def coerce(raw: str | int) -> str | int:   # correct
        return raw
"""))

`Overloaded function implementation cannot produce return type of signature 2`
— named precisely.

### When to use it

| Reach for `@overload` when | Prefer something else when |
|---|---|
| the return type genuinely depends on argument **types** | it depends on a *value* — use `Literal` overloads, or two functions |
| a `default` parameter removes the `None` (above) | you would need more than three or four stubs |
| you are typing an existing dynamic API | you are designing something new — 🔴 **two clearly-named functions usually beat one overloaded one** |

## Protocols you already use

The standard library is full of them, and knowing the names saves you inventing your own:

| Protocol | Means |
|---|---|
| `Iterable[T]` / `Iterator[T]` | `__iter__` / `__next__` (**4.3**) |
| `Sequence[T]` / `Mapping[K, V]` | indexable / key-lookup (**16.3**) |
| `SupportsIndex`, `SupportsFloat` | `__index__`, `__float__` |
| `ContextManager[T]` | `__enter__` / `__exit__` (**6.3**) |
| `Hashable`, `Sized`, `Callable` | `__hash__`, `__len__`, `__call__` |

Reach for these before writing your own — a parameter typed `Iterable[str]` communicates more,
and accepts more, than one typed `list[str]`.

In [ ]:
print(mypy("stdlib_protocols.py", r"""
    from collections.abc import Iterable, Iterator, Sized


    def total_length(items: Iterable[Sized]) -> int:
        return sum(len(item) for item in items)


    def take[T](source: Iterable[T], count: int) -> list[T]:
        result: list[T] = []
        for index, item in enumerate(source):
            if index >= count:
                break
            result.append(item)
        return result


    def numbers() -> Iterator[int]:
        yield from range(100)


    print(total_length(["build-1", "deploy-2"]))     # str is Sized
    print(total_length([[1, 2], {3, 4}]))            # so are list and set
    reveal_type(take(numbers(), 3))                  # works on a generator
    reveal_type(take("abcdef", 2))                   # ...and on a string

    total_length([1, 2, 3])                          # 🔴 int has no __len__
"""))

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Trusting `isinstance` against a `runtime_checkable` protocol.** It checks that members *exist* — not signatures, not attribute types. Both wrong classes in this notebook returned `True`.
2. **Using an `ABC` for something you do not own.** Third-party classes will never inherit from it; that is what `Protocol` is for.
3. **Returning the class name instead of `Self`.** Every subclass then reports the parent type and fluent chains break.
4. **Typing a callback as `Callable[...]` when it has keyword arguments.** `Callable` cannot express them — use a callback protocol.
5. **Writing an `@overload` implementation that cannot satisfy its stubs.** The checker catches it, but only if you read the error.
6. **Decorating the `@overload` implementation.** The final `def` must be undecorated and is invisible to callers.
7. **Overloading where two named functions would be clearer.** `read_text` and `read_bytes` beat one `read` with three stubs.
8. **Writing a protocol that already exists.** `Sized`, `Iterable`, `Hashable` and friends are in `collections.abc`.
9. **Making a protocol too large.** A ten-method protocol matches nothing; the value is in asking for exactly what you use.

## Best Practices

- Prefer `Protocol` for describing shapes, `ABC` for sharing implementation.
- Keep protocols small — name exactly the members you call, and no more.
- Use the `collections.abc` protocols before writing your own.
- Return `Self` from any method that returns the receiver or a copy of it.
- Use a callback protocol as soon as a callback has a keyword or default argument.
- Treat `runtime_checkable` as a coarse gate, and never as validation.
- Reach for `@overload` when the return type depends on argument types — and for two functions when it would take more than a few stubs.
- Name protocols for the capability (`Closeable`, `Comparable`), not the implementer.

## Practice Exercises

Try these before moving on.

1. Write a `Serialisable` protocol with `to_json`, and a class in another module that satisfies it without importing it. Confirm mypy accepts it.
2. 🔴 Take the `Report` class from this notebook and prove that `isinstance(Report(), Closeable)` is `True` while mypy rejects the same call. Which would you rather rely on?
3. Convert an `ABC` from **5.4** into a `Protocol`. What did you lose, and what did you gain?
4. Write a callback protocol for a function taking `(job_id: str, *, retries: int = 3)`, then try to express the same thing with `Callable`. What is missing?
5. Add `Self` to a fluent builder, subclass it, and confirm a chain across both classes type-checks. Then hardcode the class name and watch it break.
6. Type a `get(key)` / `get(key, default)` pair with `@overload` so the two-argument form has no `None` in its return type. Prove it with `reveal_type`.
7. 🔴 Write a generic protocol `Repository[T]` with `get`, `save` and `all`. Is it covariant or invariant in `T` (**16.3**), and why?
8. **Interview question:** what is the difference between nominal and structural typing, and which does Python's `Protocol` provide?

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | Protocols can use the PEP 695 syntax: `class Comparable[T](Protocol)` (**16.3**) |
| **3.12** | `runtime_checkable` `isinstance` checks made substantially faster, but still presence-only |
| **3.11** | 🔴 **`Self`** added (PEP 673) — before this, the idiom was a bound `TypeVar` on the method |
| **3.8** | `Protocol` and `runtime_checkable` introduced (PEP 544); `@overload` usable outside stub files |

## Where next

| Notebook | Covers |
|---|---|
| **16.5** | typing real code: classes, decorators, async, third-party stubs |
| **16.6** | adopting types in an existing codebase |

## Related

- **5.4 Duck Typing, Protocols and Composition** — the idea; this notebook is the checkable form
- **16.3 Generics** — the PEP 695 syntax used by generic protocols, and variance
- **16.2** — narrowing, which `@overload` lets callers skip
- **4.3 Generators** — `Iterable` and `Iterator`, the protocols you use most
- **6.3 Context Managers** — `__enter__`/`__exit__`, another protocol you already implement
- **15.5 Test Doubles** — a fake only has to satisfy the protocol, which is why this matters for tests